# Create impact projection files
In same format as BCD-ME inputs, one zarr per GCM (per impact). Can think about concatenating them afterwards

In [2]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar

import distributed

from funcs_support import get_filepaths,get_params,utility_save
from funcs_readregs import coeffs_mortality, coeffs_gdppc, coeffs_payroll
dir_list = get_params()

In [3]:
from distributed import Client

# Start dask client 
# (playing around with workers, threads for efficiency)
client = Client(n_workers=6,
    threads_per_worker=3,
    processes=True,
    memory_limit="8GB")
display(client)

/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 38877 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/38877/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/38877/status,Workers: 6
Total threads: 18,Total memory: 44.70 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44111,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/38877/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:37199,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/39419/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:38783,


## Default settings
(pooled regression for mortality, etc.) 

(maybe will want to concat alternate projections under a sepcification-specific dimension in the same zarrs)

In [4]:
params_proc = {'mortality':{'varname':'tassumpoly','geo_typ':'impact-regions','popwght':False},
               'gdppc':{'varname':'tassumpoly','geo_typ':'adm1','popwght':True},
               'payroll':{'varname':'days90plus','geo_typ':'adm2-us','popwght':True}}
#params_proc = {'payroll':{'varname':'days90plus','geo_typ':'adm2-us','popwght':True}}
#{'varname':'tasmaxbinF','geo_typ':'adm2-us','popwght':True}}

In [5]:
overwrite = True

for proc in params_proc:
    # Get filenames of aggregated climate predictors
    fns = np.sort(glob.glob(f'{dir_list['proc']}aggs/{params_proc[proc]['varname']}_*_{params_proc[proc]['geo_typ']}*'))
    if params_proc[proc]['popwght']:
        fns = [fn for fn in fns if re.search(r'\_popwght\.zarr',fn)]
    else:
        fns = [fn for fn in fns if not re.search(r'\_popwght\.zarr',fn)]
    fns = {re.split(r'\_',re.split(r'\/',fn)[-1])[2]:fn for fn in fns}
    
    # Load regression coefficients
    if proc == 'gdppc':
        coeffs,vcv = coeffs_gdppc(return_raw=True)
    elif proc == 'mortality':
        coeffs,vcv = coeffs_mortality(return_raw=True)
    elif proc == 'payroll':
        coeffs,vcv = coeffs_payroll(return_raw=True)
    else:
        raise KeyError()
    
    for mod in tqdm(fns):
        output_fn = f'{dir_list['proc']}impact_projs/{proc}/{re.sub(params_proc[proc]['varname'],f'd{proc}',re.split(r'\/',fns[mod])[-1])}'
        
        if overwrite or (not os.path.exists(output_fn)):
            
            # Load aggregated climate predictor
            ds = xr.open_zarr(fns[mod],consolidated=False,decode_timedelta=False)
        
            # Isolate predictor variables
            if params_proc[proc]['varname'] == 'tassumpoly':
                ds['degree'] = [f'tassumpoly_{k}' for k in ds.degree.values]
                ds = ds.rename({'degree':'pred'})
            elif params_proc[proc]['varname'] == 'tasmaxbinF':
                ds['binF'] = [f'tasmaxbinF_{k}' for k in ds.binF.values]
                ds = ds.rename({'binF':'pred'})
            elif params_proc[proc]['varname'] == 'days90plus':
                ds = ds.rename({'binF':'pred'})
            else:
                raise KeyError()
        
            # Calculate impact from linear regression model
            ds[proc] = (coeffs.coeffs*ds[params_proc[proc]['varname']]).sum('pred',skipna=False)
            # Calculate change in impact vs. GWL0.61
            ds['d'+proc] = ds[proc] - ds[proc].sel(gwl=0.61)

            # Add max gwl to coordinates, for referece
            ds = ds.assign_coords({'maxgwl':(('idv'),ds.gwl.isel(gwl=np.array([(g-1) if g != 0 else -1 
                                                              for g in (np.isnan(ds['d'+proc]).
                                                                        argmax('gwl').
                                                                        max([d for d in ds['d'+proc].sizes if d not in ['gwl','idv']])).
                                                                        load()])).values)})
            # Make sure all-nan idvs are all-nan in maxgwl
            ds['maxgwl'] = ds.maxgwl.where(~np.isnan(ds['d'+proc]).all([d for d in ds['d'+proc].sizes if d != 'idv']).load())
        
            # Prepare output 
            ds_out = ds[['locid','d'+proc]]
            ds_out.attrs = {'SOURCE':'project_impacts.ipynb',
                    'DESCRIPTION':f'changes in {proc}, at the {params_proc[proc]['geo_typ']} level',
                    'pop_weighted':params_proc[proc]['popwght']}
        
            # Export
            with warnings.catch_warnings():
                # Catches a Zarr v3 error related to string coordinates saved 
                # as objects instead of strings, I believe
                warnings.filterwarnings('ignore')
                utility_save(ds_out,output_fn,zarr_mode='w',save_kwargs = {'zarr_format':3,'consolidated':False})
        else:
            print(output_fn+' exists, skipped!')

  0%|          | 0/12 [00:00<?, ?it/s]

/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/mortality/dmortality_agg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_impact-regions.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/mortality/dmortality_agg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_impact-regions.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/mortality/dmortality_agg_CESM2-WACCM_ALLRUNS_ALLGWLs_impact-regions.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/mortality/dmortality_agg_CESM2-WACCM_ALLRUNS_ALLGWLs_impact-regions.zarr saved!


...

/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll_agg_MIROC6_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll_agg_MIROC6_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll_agg_MPI-ESM1-2-LR_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll_agg_MPI-ESM1-2-LR_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll_agg_MRI-ESM2-0_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll_agg_MRI-ESM2-0_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!


In [90]:
ds['maxgwl']

<xarray.DataArray 'maxgwl' (idv: 25)> Size: 200B
array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan,  4.,  4.,  4.,  4.,  4.,  4.])
Coordinates:
  * idv         (idv) int64 200B 0 1 2 3 4 5 6 7 8 ... 17 18 19 20 21 22 23 24
    experiment  (idv) <U6 600B 'ssp245' 'ssp245' 'ssp245' ... 'ssp585' 'ssp585'
    run         (idv) <U9 900B 'r10i1p1f1' 'r11i1p1f1' ... 'r5i1p1f1' 'r9i1p1f1'
    model       (idv) <U13 1kB 'ACCESS-ESM1-5' ... 'ACCESS-ESM1-5'
    maxgwl      (idv) float64 200B nan nan nan nan nan ... 4.0 4.0 4.0 4.0 4.0

In [84]:
mod = 'ACCESS-ESM1-5'
# Load aggregated climate predictor
ds = xr.open_zarr(fns[mod],consolidated=False,decode_timedelta=False)

# Isolate predictor variables
if params_proc[proc]['varname'] == 'tassumpoly':
    ds['degree'] = [f'tassumpoly_{k}' for k in ds.degree.values]
    ds = ds.rename({'degree':'pred'})
elif params_proc[proc]['varname'] == 'tasmaxbinF':
    ds['binF'] = [f'tasmaxbinF_{k}' for k in ds.binF.values]
    ds = ds.rename({'binF':'pred'})
elif params_proc[proc]['varname'] == 'days90plus':
    ds = ds.rename({'binF':'pred'})
else:
    raise KeyError()

# Calculate impact from linear regression model
ds[proc] = (coeffs.coeffs*ds[params_proc[proc]['varname']]).sum('pred',skipna=False)
# Calculate change in impact vs. GWL0.61
ds['d'+proc] = ds[proc] - ds[proc].sel(gwl=0.61)


# Add max gwl to coordinates, for referece
ds = ds.assign_coords({'maxgwl':(('idv'),ds.gwl.isel(gwl=np.array([(g-1) if g != 0 else -1 
                                                  for g in (np.isnan(ds['d'+proc]).
                                                            argmax('gwl').
                                                            max([d for d in ds['d'+proc].sizes if d not in ['gwl','idv']])).
                                                            load()])).values)})


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


In [ ]:
2+2

In [54]:
ds.gwl.values[[4,5,6]]

array([2.5, 3. , 3.5])

In [50]:
[g if g != 0 else -1 
                                                              for g in (np.isnan(ds['d'+proc]).
                                                                        argmax('gwl').
                                                                        max([d for d in ds['d'+proc].sizes if d not in ['gwl','idv']])).
                                                                        load()]

[<xarray.DataArray 'dpayroll' ()> Size: 8B
 array(5)
 Coordinates:
     run         <U8 32B 'r1i1p1f1'
     model       <U10 40B 'MRI-ESM2-0'
     experiment  <U6 24B 'ssp245'
     idv         int64 8B 0
     maxgwl      float64 8B 3.0,
 <xarray.DataArray 'dpayroll' ()> Size: 8B
 array(5)
 Coordinates:
     run         <U8 32B 'r2i1p1f1'
     model       <U10 40B 'MRI-ESM2-0'
     experiment  <U6 24B 'ssp245'
     idv         int64 8B 1
     maxgwl      float64 8B 3.0,
 <xarray.DataArray 'dpayroll' ()> Size: 8B
 array(4)
 Coordinates:
     run         <U8 32B 'r3i1p1f1'
     model       <U10 40B 'MRI-ESM2-0'
     experiment  <U6 24B 'ssp245'
     idv         int64 8B 2
     maxgwl      float64 8B 2.5,
 <xarray.DataArray 'dpayroll' ()> Size: 8B
 array(4)
 Coordinates:
     run         <U8 32B 'r4i1p1f1'
     model       <U10 40B 'MRI-ESM2-0'
     experiment  <U6 24B 'ssp245'
     idv         int64 8B 3
     maxgwl      float64 8B 2.5,
 <xarray.DataArray 'dpayroll' ()> Size: 8B
 array(5)